# MMF 1941 - Initial exploratory analysis
Proposal A04-03 - First progress checkpoint

## 1. Project and data context
The eventual research question compares daily-OHLC spread estimators across asset classes and liquidity regimes against an intraday benchmark. This checkpoint is **descriptive and diagnostic only**: no CS, AR, EDGE, Roll, benchmark selection, or estimator-performance claims.

Project horizon: **2015-01-01 to 2025-12-31**. Common pilot: **2024-11-04 to 2024-11-15**, aligned by recorded calendar dates with the requested Bloomberg pilot; no Bloomberg data are read here. Alignment of actual instants remains unresolved because timezone metadata are unavailable.

Raw files come exclusively from `RISKLAB_DATA_ROOT/intraday_data`. Timestamps remain naive recorded clock values; no timezone conversion or common session filter is applied. No minute bars are inserted, no prices are carried forward, and no contracts are concatenated. The final observational unit will be asset-day; these diagnostics inspect raw bars and recorded calendar dates.

**Reproduction:** run `python -B src/eda_diagnostics.py` from the project root, then run this notebook. Set `REBUILD = True` below to rescan from the notebook. The default reads generated tables to avoid another multi-gigabyte scan. Software versions and diagnostic definitions are in `outputs/diagnostics/run_manifest.json`.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "AGENTS.md").exists(), "Run from project root or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
from eda_diagnostics import run_pipeline, make_figures, make_common_window_presentation

REBUILD = False
if REBUILD:
    run_pipeline(ROOT)

TABLES = ROOT / "outputs" / "tables"
DIAG = ROOT / "outputs" / "diagnostics"
def table(name):
    return pd.read_csv(TABLES / (name + ".csv"))
coverage = table("pilot_coverage")
common = table("common_window_coverage")
quality = table("data_quality_summary")
daily = table("bars_per_calendar_day")
gaps = table("gap_summary")
desc = table("descriptive_summary")
audit = pd.read_csv(DIAG / "file_scan_audit.csv")
manifest = json.loads((DIAG / "run_manifest.json").read_text())
pd.set_option("display.max_columns", 30)
display(pd.Series({k: manifest[k] for k in ["project_start", "project_end", "common_start", "common_end", "inserted_minute_bars", "timestamp_timezone", "session"]}))

# Refresh presentation artifacts from validated diagnostics; no raw rescan needed.
make_figures(daily, table("hourly_activity"), ROOT / "outputs")
presentation_summary = make_common_window_presentation(daily, quality, common, ROOT / "outputs")


## 2. Pilot universe
Eight deliberately selected pilot instruments, not a representative or survivorship-free universe. ES_Z24 and CL_Z24 are individual contracts; their differing lifetimes are part of the coverage diagnostic. File-name adjustment labels for stocks/ETFs are recorded, not independently verified.

In [ ]:
schema = pd.read_csv(DIAG / "pilot_schema.csv")
display(schema[["asset_class", "file", "exists", "file_size_bytes"]])


## 3. Raw schema summary
The accepted small-read inspection found six comma-separated fields with no header: timestamp, open, high, low, close, volume. Field semantics are inferred from position, not provider documentation. Price fields parse numerically; crypto volume is fractional. Native price and volume units remain unresolved. The pipeline retains raw physical line numbers and separately records malformed rows.

In [ ]:
display(schema[["file", "number_of_columns", "has_header", "timestamp_format", "inferred_dtypes_json", "schema_basis"]])


## 4. Full-file date coverage within 2015-2025
Every physical row was scanned; project extrema below are calculated from all assignable in-horizon timestamps, not just file endpoints. Invalid-price and duplicate rows remain in coverage counts. The separate scan audit counts observations outside the horizon and rows that cannot be assigned to dates. **No 2026 rows enter project summaries.**

Calendar-date counts are descriptive and have no expected-session denominator. Dates with zero observations may be closures, outside a contract's life, or unavailable data; they are not automatically missing trading days.

In [ ]:
display(coverage)
display(audit)
assert (audit.raw_rows_scanned == audit[["malformed_column_rows", "unparseable_timestamp_rows", "rows_before_2015", "rows_after_2025", "project_rows"]].sum(axis=1)).all()
assert pd.to_datetime(coverage.last_timestamp).lt("2026-01-01").all()


## 5. Common-window coverage: 2024-11-04 to 2024-11-15
The inclusive 12-calendar-day window is applied to recorded date labels. Different bar counts are not coverage percentages or comparable session completeness scores.

In [ ]:
display(common)

## 6. Duplicate timestamps
Both duplicate-affected rows (all occurrences) and excess rows beyond the first occurrence are reported. Duplicate timestamps are retained. Full flags with source line numbers are exported to `outputs/diagnostics/<asset>_flagged_rows.csv`. Equal timestamps do not establish whether records are exact duplicates or distinct provider observations.

In [ ]:
display(quality[["asset", "window", "observed_rows", "duplicate_timestamp_rows", "duplicate_excess_rows"]])

## 7. Invalid OHLC rows
A finite OHLC row is flagged when a price is nonpositive, high is below low, or open/close lie outside [low, high]. This is a review flag, not a permanent deletion policy; nonpositive prices require instrument-specific interpretation. Non-finite or missing OHLC values are counted separately. Coverage keeps all assignable rows. Range and return descriptives use explicitly documented validity masks.

In [ ]:
display(quality[["asset", "window", "invalid_ohlc_rows", "non_finite_rows"]])

## 8. Missing / non-numeric values
Blank and conventional missing tokens are recorded before numeric conversion. Non-numeric tokens and infinities have separate flags. Unparseable timestamps cannot be assigned to either window and therefore appear in the full-file audit and per-asset unassignable-row files, not in a fabricated calendar date. Malformed-width records are also logged with their original raw text.

In [ ]:
display(quality[["asset", "window", "missing_value_rows", "non_numeric_rows", "non_finite_rows"]])
display(quality[["asset", "window"] + [c for c in quality if c.startswith("missing_") and c != "missing_value_rows"]])
display(audit[["asset", "malformed_column_rows", "unparseable_timestamp_rows"]])


## 9. Zero-volume observations
Zero and negative volumes are preserved and flagged. Whether zero volume represents a valid quote-like update, a trade-free bar, or a provider convention requires metadata. Raw volume magnitudes must not be interpreted as comparable liquidity across instruments.

In [ ]:
display(quality[["asset", "window", "zero_volume_rows", "negative_volume_rows"]])

## 10. Bars per calendar day
The calendar-date table contains observed row counts and distinct timestamp counts. Adding zero-count **calendar-date summary rows** is not filling missing minute bars. Duplicate raw observations remain in row counts. Both all-calendar-date and observed-date-only distributions are shown; neither labels a day complete.

Futures days with 1-10 observations are marked for sparse-data review. This transparent descriptive threshold does **not** identify an active-contract start, exclude observations, or define a continuous futures series.

**Diagnostic only:** retain the full-horizon figure for coverage review. It is not the main preliminary-result figure because ES_Z24 and CL_Z24 have limited individual-contract lifetimes.

In [ ]:
display(table("bars_per_day_summary").query("window == 'project'"))
display(Image(filename=str(ROOT / "outputs/figures/bars_per_day_by_asset.png"), width=1000))
display(table("futures_sparse_review"))
display(daily.query("window == 'project' and asset_class == 'futures' and futures_sparse_review_flag").groupby("asset", sort=False).head(8))
assert coverage.inserted_minute_bars.eq(0).all()


## 11. Bars per day in the common pilot window
Counts include every recorded clock hour and retain duplicates. Zeros on calendar dates are reported without classifying market closures or missing trading sessions.

**Main preliminary-result figure for presentation:** the common window shows all eight assets over identical recorded calendar dates, with asset classes distinguished by panel and color. It does not establish equivalent sessions. The compact table summarizes observed dates only; duplicate_rows counts all duplicate-affected records, consistent with the quality diagnostics.

In [ ]:
display(Image(filename=str(ROOT / "outputs/figures/common_window_bars_per_day.png"), width=1100))
display(presentation_summary)
display(daily.query("window == 'common'").pivot(index="date", columns="asset", values="observed_rows"))
display(table("bars_per_day_summary").query("window == 'common'"))


## 12. Distribution of timestamps by hour of day
The figure shows each asset's distribution across its own observed rows, **not coverage percentages**. Raw clock hours may refer to different timezones. Unequal coverage periods and the fixed pilot sample limit cross-asset comparisons.

For stocks/ETFs, records outside 09:30-16:00 on the unlocalized clock suggest extended-hour coverage **only if** timestamps use U.S. exchange local time. This reference marker does not assign a timezone or impose a session. Futures and crypto hours require their own metadata.

In [ ]:
display(Image(filename=str(ROOT / "outputs/figures/hourly_activity_by_asset_class.png"), width=1000))
display(table("raw_clock_coverage"))


## 13. Maximum and typical time gaps
Gaps are minutes between stable-chronologically-sorted observed records within each window. Duplicate timestamps yield zero gaps. The same-calendar-date subset is reported separately; all-adjacent gaps also include overnight intervals, weekends, closures, and sparse contract periods.

A gap larger than one minute is **not automatically a missing bar**. No minute grid or forward fill is used. Raw-order backward steps are reported in the scan audit rather than silently assuming the files were sorted.

In [ ]:
display(gaps)
display(audit[["asset", "backward_steps_in_raw_order"]])


## 14. Price, volume, intraday range and simple return descriptives
- Close and volume summaries use finite native values, including flagged numeric values; units and adjustment conventions remain unresolved.
- Bar range = high - low; normalized bar range = 100 * (high - low) / close. These use finite, positive, internally consistent OHLC rows.
- Observed-calendar-day range uses the maximum high minus minimum low among valid OHLC rows. It is not a standardized session range.
- Simple return = 100 * (close / previous observed close - 1). Returns are computed separately within each window, without filling, from adjacent chronologically ordered valid OHLC pairs with strictly increasing timestamps and no duplicate-affected endpoint.
- Adjacent-observation returns have variable elapsed time and may cross dates. Exactly-one-minute pairs are summarized separately. These are descriptive price changes, not spread estimators or standardized daily returns.

Undefined/non-finite derived values and sample sizes are reported. These diagnostic masks do not freeze the final project's cleaning policy.

In [ ]:
display(desc.query("window == 'project'"))
display(desc.query("window == 'common'"))


### Traceable manual checks
Ten deterministic examples: the first two common-window observations for AAPL and BTC, and the first for each other instrument. Each check stores the raw path, 1-based physical line number, original text, parsed fields, prior-observation reference, derived range/gap/return, and verification results. Selection is for traceability, not a random quality audit.

In [ ]:
checks = pd.read_csv(DIAG / "manual_checks.csv")
display(checks)
assert len(checks) == 10 and checks.check_pass.all()


## 15. Cross-asset descriptive comparison
Compare counts, flags and gap distributions while retaining different coverage periods. Common-window tables reduce date-span differences but do not resolve session, timezone, adjustment or unit differences. Raw price/volume levels are not ranked as liquidity measures, and no cross-class session-coverage percentages are calculated.

In [ ]:
comparison = common[["asset", "asset_class", "observed_rows", "observed_calendar_days"]].merge(
    quality.query("window == 'common'")[["asset", "duplicate_timestamp_rows", "invalid_ohlc_rows", "zero_volume_rows"]], on="asset")
comparison = comparison.merge(gaps.query("window == 'common' and population == 'all_adjacent_observed_records'")[["asset", "median", "max"]].rename(columns={"median": "median_gap_minutes", "max": "max_gap_minutes"}), on="asset")
display(comparison)


## 16. Data limitations and unresolved metadata
- Timezone, daylight-saving convention, bar timestamp meaning (start/end), session calendars and volume units are unresolved.
- OHLC positional semantics and filename adjustment labels require provider confirmation. Trade versus quote content and price currency/conversion conventions remain unverified.
- FX is expected by the inventory but physically unavailable in the mounted intraday archive; no source is substituted.
- Two selected futures contracts do not cover the entire research horizon or define continuous series. Sparse early observations must remain visible.
- The eight assets are a pilot sample; generalization and survivorship assumptions are not justified.
- Daily/EOD Bloomberg alignment has not been verified at the timestamp/session level.
- Raw data extend beyond the project horizon; out-of-window observations are counted only in the audit. Missing bars are not synthesized.
- Full-file parse/quality failures and diagnostic validity masks are documented, rather than treated as final exclusion rules.

## 17. Implications for later benchmark construction
The benchmark must be chosen from its economic meaning and available intraday fields, not from which daily estimator appears to perform best. OHLCV bars alone do not establish observed bid/ask quotes or trade direction. Provider definitions and suitable benchmark feasibility need review.

Before constructing asset-day benchmarks, resolve timezone/session mapping, trading-date assignment across midnight, return intervals, missing/zero-volume handling, and price adjustment consistency. Decide how sparse contract history and contract rolls enter the design without retrospectively defining liquid periods from estimator outcomes. Retain an auditable mapping from source rows to eventual asset-days.

## 18. Open methodological decisions
Benchmark definition; timezone and session conventions; asset-day boundaries; final data-quality exclusions; duplicate resolution; treatment of zero/negative prices and volumes; coverage criteria; liquidity-regime definitions; futures contract selection/roll conventions; and treatment of negative or undefined estimator values remain **open**. No 390-minute denominator or completeness classification is used.

### Checkpoint summary

In [ ]:
# Reconcile every persisted diagnostic collection without changing methodology.
from eda_diagnostics import validate_outputs, validate_parser
validate_parser()
all_coverage = pd.concat([coverage, common], ignore_index=True)
for col in ["first_timestamp", "last_timestamp"]:
    all_coverage[col] = pd.to_datetime(all_coverage[col])
validate_outputs({
    "audit": audit, "coverage": all_coverage, "daily": daily,
    "hourly": table("hourly_activity"), "manual": checks,
})
summary = table("common_window_presentation_summary").set_index("asset")
observed = daily.query("window == 'common' and observed_rows > 0")
expected = observed.groupby("asset").observed_rows.agg(
    observed_dates="size", total_rows="sum",
    median_rows_per_observed_day="median", min_rows_per_observed_day="min",
    max_rows_per_observed_day="max")
pd.testing.assert_frame_equal(summary[expected.columns].sort_index(), expected.sort_index(),
                              check_dtype=False)
q = quality.query("window == 'common'").set_index("asset")
for target, source in [("duplicate_rows", "duplicate_timestamp_rows"),
                       ("invalid_ohlc_rows", "invalid_ohlc_rows"),
                       ("zero_volume_rows", "zero_volume_rows")]:
    assert summary[target].sort_index().equals(q[source].sort_index())
assert len(summary) == 8 and summary.total_rows.sum() == common.observed_rows.sum()
assert pd.read_csv(DIAG / "failures.csv").empty
print("All diagnostics reconcile; presentation table: 8 assets, 91,875 rows; 10 manual checks passed.")


In [ ]:
print((DIAG / "checkpoint_summary.txt").read_text())
print("\nUnresolved: timezone/session metadata; price/volume units and adjustments; sparse futures history and contract policy; unavailable FX; benchmark definition and final cleaning/estimator-value rules.")


## Checkpoint presentation generation

Raw data -> EDA pipeline -> CSV/PNG outputs -> supplied PowerPoint template -> final progress deck.

The builder reads the accepted diagnostics and Bloomberg source-log metadata. It performs no new empirical analysis or estimator calculations. The template is required and remains unchanged. The generated PowerPoint is saved separately; no binary deck is embedded in this notebook.


In [ ]:
from build_presentation import build_presentation
presentation_path = build_presentation(ROOT)
print(presentation_path.relative_to(ROOT))
